samples <- c(
  "GSM6202450",
  "GSM6202452",
  "GSM6202453",
  "GSM6202455",
  "GSM6202456",
  "GSM6202457",
  "GSM6202458",
  "GSM6202459",
  "GSM8101532",
  "HCAHeart8102868"
)
 removed because of other cell types

In [ ]:
# QC, scVI integration and myeloid subsetting of the heart single-cell samples.
import os
import pandas as pd
import scanpy as sc
import anndata as ad

input_dir = "/home/yanjun3/atlas/heart_sc/"
os.listdir(input_dir)

In [ ]:
test = sc.read(f"{input_dir}/HCAHeart7905327.h5ad")

In [ ]:
print(test.X)

In [ ]:
input_dir = "/home/yanjun3/atlas/heart_sc"

files = [f for f in os.listdir(input_dir) if f.endswith(".h5ad")]

adatas, keys = [], []

for fname in files:
    file_path = os.path.join(input_dir, fname)
    adata = sc.read(file_path)
    sc.pp.filter_cells(adata, min_genes=300)
    sc.pp.filter_genes(adata, min_cells=10)
    adata.layers["counts"] = adata.X.copy()
    sc.pp.normalize_total(adata, target_sum=1e4)
    sc.pp.log1p(adata)
    adata.layers["log_counts"] = adata.X.copy()

    adata.write(file_path)

    if adata.n_obs == 0:
        continue
    sample = fname.replace(".h5ad", "")
    adata.obs["sample"] = sample
    adatas.append(adata)
    keys.append(sample)

heart_sn_adata = sc.concat(adatas, label="sample", keys=keys, index_unique="-", join="outer")
info = pd.read_csv("/home/yanjun3/atlas/heart_sc/heart_sc_clean.csv").drop_duplicates(subset=["sample_id"])
heart_sn_adata.obs = heart_sn_adata.obs.join(info.set_index("sample_id"), on="sample")

heart_sn_adata.write(os.path.join(input_dir, "heart_sc_qc1.h5ad"))
heart_sn_adata.obs

In [ ]:
del heart_sn_adata

In [ ]:
del adatas

In [ ]:
import os
import pandas as pd
import scanpy as sc
import anndata as ad

import scvi
import seaborn as sns
import torch
from rich import print
from scib_metrics.benchmark import Benchmarker

In [ ]:
adata = sc.read("/home/yanjun3/atlas/heart_sc/heart_sc_qc1.h5ad")

In [ ]:
adata.obs_names_make_unique()

In [ ]:
adata

In [ ]:
sc.settings.verbosity = 0
sc.settings.set_figure_params(
    dpi=300,
    facecolor="white",
    frameon=True,  
    format="pdf"   
)

In [ ]:
sc.set_figure_params(figsize=(6, 6), frameon=False, dpi = 300)
sc.settings.figdir = "/home/yanjun3/atlas/heart_sc/figure_qc1"
sns.set_theme()
torch.set_float32_matmul_precision("high")

%config InlineBackend.print_figure_kwargs={"facecolor": "w"}
%config InlineBackend.figure_format="retina"

In [ ]:
batch_key = "sample"

In [ ]:
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True, log1p=True)

In [ ]:
sc.pl.violin(
    adata,
    ["n_genes_by_counts", "total_counts", "pct_counts_mt"],
    jitter=0.4,
    multi_panel=True,
)

In [ ]:
adata

In [ ]:
adata = adata[
    (adata.obs["n_genes_by_counts"] >= 500) &
    (adata.obs["n_genes_by_counts"] <= 10_000) &
    (adata.obs["total_counts"] >= 800) &
    (adata.obs["total_counts"] <= 5_000) &
    (adata.obs["pct_counts_mt"] < 5)
]
adata

In [ ]:
adata.obs["anatomical_site"] = adata.obs["anatomical_site"].str.lower()
adata.obs["Disease"] = adata.obs["Disease"].str.lower()

In [ ]:
adata

In [ ]:
sc.pp.highly_variable_genes(adata, n_top_genes = 8000)
adata_hvg = adata[:, adata.var["highly_variable"]].copy()
adata_hvg 

In [ ]:
scvi.model.SCVI.setup_anndata(
    adata_hvg,
    layer="counts",
    batch_key=batch_key,
    continuous_covariate_keys=[
        "pct_counts_mt",
        "total_counts",
        "n_genes_by_counts"
    ]
)

In [ ]:
model_scvi = scvi.model.SCVI(adata_hvg)
model_scvi

In [ ]:
model_scvi.view_anndata_setup()

In [ ]:
model_scvi = scvi.model.SCVI(adata_hvg)
model_scvi

In [ ]:
import numpy as np
max_epochs_scvi = np.min([round((20000 / adata.n_obs) * 400), 400])
max_epochs_scvi

In [ ]:
model_scvi.train()

In [ ]:
adata.obsm["X_scVI"] = model_scvi.get_latent_representation()
#adata.layers["scvi_normalized"] = model_scvi.get_normalized_expression()

In [ ]:
adata.write('heart_sc_qc1_scvi.h5ad')

In [ ]:
adata = sc.read('heart_sc_qc1_scvi.h5ad')

In [ ]:
sc.pp.neighbors(adata, use_rep="X_scVI", n_neighbors=15)

In [ ]:
sc.tl.umap(adata, min_dist=0.6,  spread=2.0)

In [ ]:
sc.pl.umap(
    adata,
    color=["project_id"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data'
)

In [ ]:
sc.tl.leiden(adata, key_added="leiden_res0_3", resolution=0.3)
sc.pl.umap(
    adata,
    color=["leiden_res0_3"]
    ,
    frameon=False,
    ncols=1, size = 30, legend_loc = 'on data'
)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby="leiden_res0_3", method="wilcoxon")
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, standard_scale='var', 
                               # save = 'res_0_5_topmarkers'
                               )

In [ ]:
sc.pl.rank_genes_groups_dotplot(adata, n_genes=10, standard_scale='var', 
                                save = 'qc1_res_0_3_topmarkers'
                               )

In [ ]:

marker_dict = {
    "cd14 monocytes": [
        "S100A9", "S100A8", "LYZ", "VCAN", "NAMPT", "S100A4",
        "TYROBP", "SRGN", "FCER1G", "PLAUR", "CTSS", "S100A12",
        "COTL1", "SAMSN1", "KYNU", "AIF1", "SAT1", "FTL", "FCN1",
    ],

    "cd16 monocytes": [
        "FCER1G", "TYROBP", "S100A9", "AIF1", "SAT1", "LST1",
        "FTL", "COTL1", "CTSS", "LYZ", "S100A4", "S100A8",
        "NAMPT", "FCN1", "FTH1", "HLA-DRA", "CYBA", "SRGN",
        "FCGR3A",
    ],

    "cDC1s": [
        "HLA-DRA", "HLA-DPB1", "HLA-DPA1", "CD74", "LYZ",
        "HLA-DQA1", "HLA-DRB1", "CCSER1", "HLA-DQB1", "CPVL",
        "HDAC9", "WDFY4", "IDO1", "BASP1", "CST3", "HLA-DRB5",
        "SAT1", "PTPRC", "CLNK",
    ],

    "cDC2s": [
        "HLA-DRA", "HLA-DPB1", "HLA-DRB1", "HLA-DPA1", "CD74",
        "HLA-DQA1", "HLA-DQB1", "LYZ", "TYROBP", "FCER1G",
        "SRGN", "AIF1", "MS4A6A", "CST3", "CD83", "FCER1A",
        "HLA-DMA", "SIPA1L1", "AREG",
    ],

    "cycling myeloid": [
        "C1QB", "LYZ", "C1QA", "S100A4", "CD74", "HLA-DRA",
        "C1QC", "TUBA1B", "AIF1", "STMN1", "MS4A6A", "HMGB2",
        "FCER1G", "CTSB", "CD163", "TYROBP", "HLA-DRB1",
        "F13A1", "RBPJ",
    ],

    "lyve1 macrophages": [
        "C1QA", "HLA-DRA", "C1QB", "CD74", "TYROBP", "C1QC",
        "FTL", "MS4A6A", "CD163", "F13A1", "CTSB", "AIF1",
        "HLA-DRB1", "RBPJ", "MRC1", "FCER1G", "LYZ",
        "HLA-DPB1", "RBM47",
    ],

    "spp1 macrophages": [
        "FTL", "CTSB", "TYROBP", "FTH1", "NAMPT", "PLAUR",
        "FCER1G", "CXCL8", "SLC16A10", "KYNU", "CD44", "LYZ",
        "S100A4", "SIPA1L1", "CXCL3", "LAPTM5", "CD163",
        "HLA-DRA", "CTSD",
    ],

    "TMEM163 macrophages": [
        "HLA-DRA", "CD74", "HLA-DPA1", "HLA-DPB1", "HLA-DRB1",
        "HLA-DQA1", "TYROBP", "C1QA", "C1QB", "C1QC",
        "HLA-DQB1", "CCL3", "CD83", "CCL4", "IER3", "SAT1",
        "CCL3L1", "FCGR2A", "AIF1", "FTL",
    ],
}

genes = [
    "TYROBP",
    "LYZ",
    "FTL",
    "HLA-DRA",
    "FCER1G",
    "AIF1",
    "NAMPT",
    "CTSB",
    "SAT1",
    "S100A4",
    "CTSS",
    "SRGN",
    "S100A9",
    "CD74",
    "SIPA1L1",
    "FTH1",
    "LAPTM5",
    "RBM47",
    "CD163",
    "MS4A6A",
]


In [ ]:
adata

In [ ]:
sc.pl.dotplot(
    adata,
    var_names = marker_dict,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
    save = 'disco_fine.png'
)


In [ ]:
sc.pl.dotplot(
    adata,
    var_names = genes,
    groupby   = "leiden_res0_3",   # change to your obs column
    dendrogram = False,
    save = 'disco_broad.png'
)


In [ ]:
adata

In [ ]:
clusters = ["1", "14", "17","28"]
cluster_key = "leiden_res0_3"

adata_sub = adata[adata.obs[cluster_key].isin(clusters)].copy()
adata_sub

In [ ]:
adata_sub.write('qc2_myeloid.h5ad')

In [ ]:
result = adata.uns['rank_genes_groups']
groups = result['names'].dtype.names
df = pd.DataFrame(
    {f"{group}_{key}": result[key][group]
     for group in groups for key in ['names', 'scores', 'pvals', 'pvals_adj', 'logfoldchanges']}
)
df.to_csv(f"{input_dir}/v1_res03.csv", index=False)

In [ ]:
sc.pl.violin(
    adata_hvg,
    keys=["pct_counts_mt"],
    groupby="leiden_res0_3",
    stripplot=False,
    jitter=0,
    rotation=45,
    multi_panel=True,
    save = 'mt_0_3.png'
)

In [ ]:
sc.pl.scatter(adata_hvg, "total_counts", "n_genes_by_counts", color="pct_counts_mt")

In [ ]:
clusters = ["4", "7", "11"]         
cluster_key = "leiden_res0_3"          
# cells in those clusters (from adata_hvg)
sel_cells = adata_hvg.obs_names[adata_hvg.obs[cluster_key].astype(str).isin(clusters)]

# mask onto your full adata, then subset
adata.obs["mask"] = adata.obs_names.isin(sel_cells)
adata_subset = adata[adata.obs["mask"]].copy()
adata_subset.write('/home/yanjun3/atlas/heart_sn//group1_myeloid.h5ad')

In [ ]:
sc.pl.dotplot(
    adata_mac_hvg, 
    var_names=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2'],
    groupby='leiden_res0_8',
    dot_max = 0.8,
    standard_scale='var', 
    color_map='RdBu_r' ,
    
   save = 'scvi_08.png'
)

In [ ]:
sc.pl.umap(
    adata_mac_hvg,
    color=['LYVE1', 'SELENOP', 'F13A1', 'MRC1', 'TIMD4', 'FOLR2', 
          'HLA-DQA1', 'HLA-DQB1', 'leiden_res0_8'],
    frameon=False,
    ncols=3, size = 30, legend_loc = 'on data', color_map = 'RdBu_r',
    save = 'tmp_umap'
)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import gseapy as gp

adata_obj = adata_mac_hvg          # your object
grp      = "leiden_res0_8"
targets  = {"0","3","5","11","14"}
combo    = "c0_3_5_11_14_vs_rest"
de_key   = "deg_combo"
gene_sets= "MSigDB_Hallmark_2020"

# 1) make two-level categorical label: target vs other
mask = adata_obj.obs[grp].astype(str).isin(targets)
adata_obj.obs[combo] = pd.Categorical(
    np.where(mask, "target", "other"),
    categories=["other", "target"]   # ensures both levels exist
)

# 2) DE: target vs other
sc.tl.rank_genes_groups(
    adata_obj, groupby=combo, groups=["target"], reference="other",
    method="wilcoxon", use_raw=False, key_added=de_key
)

# 3) build preranked list and run GSEA (tutorial style)
deg = sc.get.rank_genes_groups_df(adata_obj, group="target", key=de_key)
rnk = (deg[["names","logfoldchanges"]]
       .dropna().drop_duplicates("names")
       .set_index("names")["logfoldchanges"]
       .sort_values(ascending=False))

pre = gp.prerank(
    rnk=rnk, gene_sets=gene_sets, outdir=None,
    min_size=10, max_size=5000, permutation_num=1000,
    seed=42, threads=4          # use threads (processes is deprecated)
)

res = pre.res2d.copy()          # has columns like Term, NES, FDR q-val
res_up = res[res["NES"] > 0].sort_values("NES", ascending=False)

# (optional) quick plots
gp.barplot(res_up, column="NES", title=f"Upregulated terms: {combo}")
gp.dotplot(res_up, column="FDR q-val", title=f"Upregulated terms: {combo}")


In [ ]:
import scanpy as sc
import gseapy as gp
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# ---- inputs you set ----
adata_obj   = adata_mac_hvg          # your AnnData (HVG or all-genes)
group_var   = "leiden_res0_8"        # cluster column
target_lab  = "3"                    # e.g., cluster 3 (or "target" if you merged clusters)
de_key      = "deg_for_enrichr"      # where DE is stored
libraries   = ["MSigDB_Hallmark_2020", "GO_Biological_Process_2021", "Reactome_2016"]
organism    = "Human"                # "Mouse" if mouse symbols

# 1) DE: target vs rest (wilcoxon)
sc.tl.rank_genes_groups(
    adata_obj, groupby=group_var, groups=[target_lab], reference="rest",
    method="wilcoxon", use_raw=False, key_added=de_key
)

# 2) Build UP/DOWN gene lists (adjust thresholds as needed)
deg = sc.get.rank_genes_groups_df(adata_obj, group=target_lab, key=de_key).dropna()
padj_thr, lfc_thr, top_n = 0.05, 0.25, 200

deg_up = (deg.query("pvals_adj < @padj_thr and logfoldchanges >  @lfc_thr")
            .sort_values(["pvals_adj","logfoldchanges"], ascending=[True, False]))
deg_dn = (deg.query("pvals_adj < @padj_thr and logfoldchanges < -@lfc_thr")
            .sort_values(["pvals_adj","logfoldchanges"], ascending=[True, True]))

up_genes = deg_up["names"].drop_duplicates().head(top_n).tolist()
dn_genes = deg_dn["names"].drop_duplicates().head(top_n).tolist()

print(f"UP genes: {len(up_genes)}, DOWN genes: {len(dn_genes)}")

# 3) Enrichr ORA (GSEApy)
enr_up = gp.enrichr(
    gene_list=up_genes,
    gene_sets=libraries,
    organism=organism,
    outdir=None, # keep in-memory
    cutoff=1.0   # don't filter; we'll sort ourselves
)
enr_dn = gp.enrichr(
    gene_list=dn_genes,
    gene_sets=libraries,
    organism=organism,
    outdir=None,
    cutoff=1.0
)

# 4) Quick view: top terms per library (UP)
def show_top(enr_res, title, n=10):
    df = enr_res.res2d.copy()
    # unify column names a bit
    df.columns = [c.replace("_", " ") for c in df.columns]
    keep = df.rename(columns={
        "Adjusted P-value":"Adjusted P-value",
        "Combined Score":"Combined Score",
        "Term":"Term",
        "Gene_set":"Gene set"
    })
    print(f"\n{title}")
    for lib in keep["Gene set"].unique():
        sub = keep[keep["Gene set"]==lib].sort_values("Adjusted P-value").head(n)
        print(f"\n— {lib}")
        print(sub[["Term","Adjusted P-value","Combined Score"]].to_string(index=False))

show_top(enr_up, f"UP in {group_var}={target_lab}")
show_top(enr_dn, f"DOWN in {group_var}={target_lab}")

# 5) Tutorial-style plots
# Barplot and dotplot for UP (filter one library or stack all)
up_df = enr_up.res2d.copy()
lib_focus = "MSigDB_Hallmark_2020"   # change per plot
up_df = up_df[up_df["Gene_set"]==lib_focus].sort_values("Adjusted P-value").head(15)

gp.barplot(
    up_df,
    title=f"Enrichr (UP): {group_var}={target_lab} — {lib_focus}",
    column="Adjusted P-value"
)
gp.dotplot(
    up_df,
    title=f"Enrichr (UP): {group_var}={target_lab} — {lib_focus}",
    column="Adjusted P-value",
    cmap=None, size=6
)

# (Optional) same plots for DOWN
dn_df = enr_dn.res2d.copy()
dn_df = dn_df[dn_df["Gene_set"]==lib_focus].sort_values("Adjusted P-value").head(15)
gp.barplot(dn_df, title=f"Enrichr (DOWN): {group_var}={target_lab} — {lib_focus}",
           column="Adjusted P-value")
gp.dotplot(dn_df, title=f"Enrichr (DOWN): {group_var}={target_lab} — {lib_focus}",
           column="Adjusted P-value", cmap=None, size=6)


In [ ]:
adata_mac_hvg.obs

In [ ]:
obs_df = adata_mac_hvg.obs[['Disease', 'leiden_res0_8']]
frequency_df = obs_df.groupby(['Disease', 'leiden_res0_8']).size().unstack(fill_value=0)
percentage_df = frequency_df.div(frequency_df.sum(axis=1), axis=0) * 100
percentage_df = percentage_df.iloc[::-1]

fig = plt.figure(figsize=(5, 3))
ax = percentage_df.plot(kind='barh', stacked=True, ax=fig.gca())  

plt.title("Proportion of clusters in obese / lean")
plt.xlabel("Percentage (%)")
plt.ylabel("sample")
plt.legend(title='Clusters', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(False)  
plt.xticks(rotation=0)  

plt.tight_layout()
plt.tight_layout()
plt.savefig("samplefreq.pdf", dpi=600)
plt.show()
